# 11 — Target Attainment Forecast

Projects each executive-set target (wait, completion, no-show) to its horizon date via a trend
line with a confidence band, and reports **on-track / at-risk / off-track**.

**Outputs the `target_attainment` insight** consumed by the Executive dashboard.

## 1. Setup & daily metrics

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")
plt.rcParams.update({"axes.titleweight": "bold", "axes.titlesize": 12, "figure.dpi": 110})
pd.set_option("display.max_columns", 40)

# QMe Now palette (matches the admin dashboards)
NAVY, STEEL, TEAL, RED, GOLD = "#2F5063", "#6E8AA6", "#2E7387", "#B23A4E", "#9A6B2E"
BLUES = sns.light_palette(NAVY, n_colors=6, reverse=True)

BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BASE))
DOW = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]
def hour_label(h): return f"{((int(h) + 11) % 12) + 1}{'am' if h < 12 else 'pm'}"
print("Ready.")

In [ ]:
from scripts import forecast_targets as ft

conn = ft.connect()
daily, targets = ft.load_daily(conn)
biz = daily["business_id"].mode().iloc[0]
dfb = daily[daily["business_id"] == biz].sort_values("visit_date")
t = targets[biz]
print(f"{dfb['business_name'].iloc[0]} · {len(dfb)} days · target wait {t['target_wait_minutes']}m, "
      f"completion {t['target_completion_rate']}%, no-show {t['target_no_show_rate']}%")
dfb[["visit_date", "avg_wait", "completion_rate", "no_show_rate"]].tail()

## 2. Trend projection per metric\nEach chart: the daily series, its linear trend projected to the horizon, and the target line.

In [ ]:
horizon = int((t.get("horizon_months") or 6) * 30)
metrics = [("avg_wait", "Average Wait (min)", t["target_wait_minutes"], True),
           ("completion_rate", "Completion Rate (%)", t["target_completion_rate"], False),
           ("no_show_rate", "No-Show Rate (%)", t["target_no_show_rate"], True)]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (col, title, goal, lower) in zip(axes, metrics):
    s = dfb.dropna(subset=[col])
    x = (s["visit_date"] - s["visit_date"].min()).dt.days.to_numpy(float)
    y = s[col].to_numpy(float)
    slope, icpt = np.polyfit(x, y, 1)
    fx = np.linspace(0, x.max() + horizon, 50)
    ax.scatter(x, y, s=12, color=STEEL, alpha=0.5)
    ax.plot(fx, slope * fx + icpt, color=NAVY, lw=2.4, label="Trend → horizon")
    ax.axhline(goal, color=RED, ls="--", lw=1.6, label=f"Target {goal}")
    ax.axvline(x.max(), color="#999", ls=":", lw=1)
    ax.set_title(title); ax.set_xlabel("Day index"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 3. Verdict table

In [ ]:
rows = []
for col, title, goal, lower in metrics:
    proj = ft.project(dfb, col, horizon)
    row = ft.assess(title, proj, goal, lower)
    if row: rows.append(row)
verdict = pd.DataFrame(rows)[["metric", "current", "projected", "confidence_band", "target", "status", "trend"]]

def color_status(v):
    return f"color: {'#2E7387' if v=='on_track' else GOLD if v=='at_risk' else RED}; font-weight:700"
verdict.style.map(color_status, subset=["status"])

In [ ]:
if os.getenv("WRITE_DB") == "1":
    ins, gen, stale = ft.build_insights(daily, targets)
    ft.upsert_insights(conn, ins, gen, stale, ft.MODEL_VERSION)
    print(f"Upserted {len(ins)} target_attainment insight(s).")
else:
    print("Preview only — set WRITE_DB=1 to persist.")
conn.close()